In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib

x_train = joblib.load('x_train.joblib')
y_train = joblib.load('y_train.joblib')
x_test = joblib.load('x_test.joblib')
y_test = joblib.load('y_test.joblib')
le = joblib.load('label_encoder.pkl')
tfidf = joblib.load('tfidf_vectorizer.pkl')

# Convert sparse matrices to tensors
x_train = torch.tensor(x_train.toarray(), dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
x_test = torch.tensor(x_test.toarray(), dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

In [2]:
class SentimentModel(nn.Module): # ann model 150 input features 4 output class
    def __init__(self):
        super(SentimentModel, self).__init__()
        self.f1 = nn.Linear(150, 50)
        self.f2 = nn.Linear(50, 20)
        self.f3 = nn.Linear(20, 4)

    def forward(self, x):
        x = torch.relu(self.f1(x))
        x = torch.relu(self.f2(x))
        return self.f3(x)

model = SentimentModel()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [3]:
# Training Loop
epochs = 120
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(x_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

Epoch [5/120], Loss: 1.3500
Epoch [10/120], Loss: 1.2220
Epoch [15/120], Loss: 1.0298
Epoch [20/120], Loss: 0.8507
Epoch [25/120], Loss: 0.7550
Epoch [30/120], Loss: 0.7274
Epoch [35/120], Loss: 0.7054
Epoch [40/120], Loss: 0.6870
Epoch [45/120], Loss: 0.6782
Epoch [50/120], Loss: 0.6715
Epoch [55/120], Loss: 0.6638
Epoch [60/120], Loss: 0.6590
Epoch [65/120], Loss: 0.6547
Epoch [70/120], Loss: 0.6506
Epoch [75/120], Loss: 0.6470
Epoch [80/120], Loss: 0.6435
Epoch [85/120], Loss: 0.6402
Epoch [90/120], Loss: 0.6368
Epoch [95/120], Loss: 0.6337
Epoch [100/120], Loss: 0.6305
Epoch [105/120], Loss: 0.6276
Epoch [110/120], Loss: 0.6274
Epoch [115/120], Loss: 0.6233
Epoch [120/120], Loss: 0.6191


In [4]:
# Evaluation
model.eval()
test_loss = 0
correct = 0

with torch.no_grad():
    for data, target in zip(x_test, y_test):
        # unsqueeze to make it [1, 100]
        output = model(data.unsqueeze(0))
        test_loss += criterion(output, target.unsqueeze(0)).item()

        # argmax(dim=1) works on the [1, 4] output
        pred = output.argmax(dim=1)
        correct += (pred == target).sum().item()

test_loss /= len(x_test)
print(f'\nTest set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{len(x_test)} ({100. * correct / len(x_test):.0f}%)')


Test set: Average loss: 0.6784, Accuracy: 7435/10219 (73%)


In [5]:

text = "i want to kill you"
text_tfidf = tfidf.transform([text])
text_tensor = torch.tensor(text_tfidf.toarray(), dtype=torch.float32)

model.eval()
with torch.no_grad():
    prediction_logits = model(text_tensor)
    predicted_idx = torch.argmax(prediction_logits, dim=1).item()
    print(f"Prediction for '{text}': {le.inverse_transform([predicted_idx])[0]}")

Prediction for 'i want to kill you': Suicidal


In [6]:
torch.save(model.state_dict(), 'sentiment_model.pth')